# Soft Delete Pattern

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/LineageLogic/LakeLogic/blob/main/examples/02_core_patterns/soft_delete/soft_delete_pattern.ipynb) 
[![GitHub Repo](https://img.shields.io/badge/GitHub-Repo-blue?logo=github)](https://github.com/LineageLogic/LakeLogic/blob/main/examples/02_core_patterns/soft_delete/soft_delete_pattern.ipynb)

## Business Scenario

In regulated environments, hard deletes remove audit history. You need a consistent soft-delete strategy that preserves lineage and supports compliance.

## Value Proposition

- Maintain audit trails without losing records
- Enable easy recovery after mistaken deletes
- Standardize delete metadata across datasets

---

## Goals

1. Flag deleted records
2. Capture deletion timestamps and reasons
3. Keep historical visibility


## 🚀 Step 1: Create the Standardized Contract

We define the standardized system columns in the materialization section.

In [ ]:
contract_yaml = """
version: 1.0.0
dataset: users_silver

source:
  type: table
  cdc_op_field: "op"          
  cdc_delete_values: ["D"]   

primary_key: ["user_id"]

materialization:
  strategy: merge
  path: "./data/users_silver/"
  format: parquet
  soft_delete_column: "_lakelogic_is_deleted"
  soft_delete_value: true
  soft_delete_time_column: "_lakelogic_deleted_at"
  soft_delete_reason_column: "_lakelogic_delete_reason"
"""

with open('contract.yaml', 'w') as f:
    f.write(contract_yaml)

print("✅ Standardized Contract created!")

## ▶️ Step 2: Initial Load

Start with active users.

In [ ]:
from lakelogic.core.processor import DataProcessor
import polars as pl
import os
import shutil

# Clean up previous runs if any
if os.path.exists("./data/users_silver/"):
    shutil.rmtree("./data/users_silver/")

df_v1 = pl.DataFrame([
    {"user_id": 1, "name": "Alice", "op": "I"},
    {"user_id": 2, "name": "Bob", "op": "I"}
])

processor = DataProcessor(contract="contract.yaml", engine="polars")
processor.run(df_v1, materialize=True)

print("\n📂 Silver Table Contents (V1):")
print(pl.read_parquet("./data/users_silver/data.parquet"))

## 🛑 Step 3: Trigger Soft Delete with Metadata

When Bob is deleted, LakeLogic will now automatically populate the `_at` and `_reason` columns.

In [ ]:
df_v2 = pl.DataFrame([
    {"user_id": 2, "name": "Bob", "op": "D"} # Bob is out
])

processor.run(df_v2, materialize=True)

# Final result check
final_df = pl.read_parquet("./data/users_silver/data.parquet")
print("\n📂 Final Silver Table with Metadata:")
print(final_df.select(["user_id", "name", "_lakelogic_is_deleted", "_lakelogic_deleted_at", "_lakelogic_delete_reason"]))
